In [1]:
import torch
import torch.nn as nn
import os
from torch.utils.data import Dataset
import transformers
from transformers import GPT2Tokenizer, GPT2LMHeadModel


c:\Users\satyam prasad\Desktop\columbina\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [3]:
from pathlib import Path

data_path=Path("data/")
text_path=data_path/"tinystories.txt"

# Open the file
with open(text_path, 'r', encoding ='utf-8') as f:
    data1 = f.read()
text_len1=len(data1)
print(text_len1)

split=int(0.0001*text_len1)
data=data1[:split]
text_len=len(data)
print(len(data))

1902092922
190209


In [4]:
#train and test split
train_split=int(0.8*text_len)
train_text=data[:train_split]
test_text=data[train_split:]
print(len(train_text), len(test_text))

152167 38042


In [ ]:
#not implimented now

with open(data_path/"train.txt", 'w', encoding ='utf-8') as f:
    f.write(train_text)
with open(data_path/"test.txt", 'w', encoding ='utf-8') as f:
   f.write(test_text)

In [ ]:
#dont use this code right now need to fix

import os
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

# Configuration
train_input_file_path = "data/train_text.txt"
test_input_file_path= "data/test_text.txt"
train_output_file_path = "data/train.bin"
test_output_file_path = "data/test.bin"
chunk_size = 1024 * 1024  # Read 1MB of text at a time (adjust based on RAM)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

print(f"Processing {train_input_file_path}...")

train_all_tokens = []
test_all_tokens = []

with open(train_input_file_path, 'r', encoding='utf-8') as f:
    while True:
        text_chunk = f.read(chunk_size)
        if not text_chunk:
            break
        

        enc = tokenizer.encode(text_chunk)
        train_all_tokens.extend(enc)
        
        # Optional: Print progress
        if len(train_all_tokens) % 100_000 == 0:
            print(f"Processed {len(train_all_tokens)} tokens...")


# 2. Convert to numpy array and save to disk
print(f"Saving {len(train_all_tokens)} tokens to {train_output_file_path}...")
token_array = np.array(train_all_tokens, dtype=np.uint16)

# Save as raw binary
with open(train_output_file_path, 'wb') as f:
    f.write(token_array.tobytes())

print("Done! File saved.")

print(f"Processing {test_input_file_path}...")
with open(test_input_file_path, 'r', encoding='utf-8') as f:
    while True:
        text_chunk = f.read(chunk_size)
        if not text_chunk:
            break
        

        enc = tokenizer.encode(text_chunk)
        test_all_tokens.extend(enc)
        
        # Optional: Print progress
        if len(test_all_tokens) % 100_000 == 0:
            print(f"Processed {len(test_all_tokens)} tokens...")
# 2. Convert to numpy array and save to disk
print(f"Saving {len(test_all_tokens)} tokens to {test_output_file_path}...")
token_array = np.array(test_all_tokens, dtype=np.uint16)

# Save as raw binary
with open(test_output_file_path, 'wb') as f:
    f.write(token_array.tobytes())
print("Done! File saved.")






#not usable right now need to fix


okay,so this part is done now time to make datasets 

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", use_fast=True)
class TokenizedDataset(Dataset):
    def __init__(self, raw_text, tokenizer, block_size=128):
        
        print("Tokenizing data... this may take a moment.")
        self.tokens = tokenizer.encode(raw_text)
        self.block_size = block_size
        
        
        self.total_chunks = len(self.tokens) // block_size

    def __len__(self):
        return self.total_chunks

    def __getitem__(self, idx):

        start_idx = idx * self.block_size
        end_idx = start_idx + self.block_size
        
        
        chunk = self.tokens[start_idx:end_idx]
        
        
        input_ids = torch.tensor(chunk, dtype=torch.long)
        
        
        attention_mask = torch.ones_like(input_ids)
        
        return input_ids, attention_mask




block_size = 128  
train_ds = TokenizedDataset(train_text, tokenizer, block_size=block_size)
test_ds = TokenizedDataset(test_text, tokenizer, block_size=block_size)

print(f"Train dataset size: {len(train_ds)} blocks")
print(f"Test dataset size: {len(test_ds)} blocks")


train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)


for batch in train_loader:
    input_ids, attention_mask = batch
    print("\nBatch Shape:", input_ids.shape)
    print("Sample Input IDs:", input_ids[0][:10]) #
    
import torch
from torch.utils.data import Dataset, DataLoader

class TokenizedDataset(Dataset):
    def __init__(self, raw_text, tokenizer, block_size=128):
        
        print("Tokenizing data... this may take a moment.")
        self.tokens = tokenizer.encode(raw_text)
        self.block_size = block_size
        
        
        self.total_chunks = len(self.tokens) // block_size

    def __len__(self):
        return self.total_chunks

    def __getitem__(self, idx):
        
        start_idx = idx * self.block_size
        end_idx = start_idx + self.block_size
        
        
        chunk = self.tokens[start_idx:end_idx]
        
        
        input_ids = torch.tensor(chunk, dtype=torch.long)
        
        
        attention_mask = torch.ones_like(input_ids)
        
        return input_ids, attention_mask




block_size = 128  


train_ds = TokenizedDataset(train_text, tokenizer, block_size=block_size)
test_ds = TokenizedDataset(test_text, tokenizer, block_size=block_size)

print(f"Train dataset size: {len(train_ds)} blocks")
print(f"Test dataset size: {len(test_ds)} blocks")

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)


for batch in train_loader:
    input_ids, attention_mask = batch
    print("\nBatch Shape:", input_ids.shape)
    print("Sample Input IDs:", input_ids[0][:10]) 
    
    
    print("\nDecoded Text Check:")
    print(tokenizer.decode(input_ids[0]))
    break

Tokenizing data... this may take a moment.


: 

In [ ]:

#dont use this code right now need to fix


import torch
import numpy as np
from torch.utils.data import Dataset

class MemmapDataset(Dataset):
    def __init__(self, bin_file, block_size=128):
        self.block_size = block_size
        
        # 1. Load the data using memory mapping
        # 'r' mode ensures we don't accidentally overwrite data
        # shape=(-1) simply means "whatever the length is"
        self.data = np.memmap(bin_file, dtype=np.uint16, mode='r')
        
        print(f"Loaded dataset with {len(self.data)} tokens.")
        
    def __len__(self):
        # We return how many blocks fit in the data
        return len(self.data) // self.block_size - 1

    def __getitem__(self, idx):
        # 2. Retrieve a slice from the disk (OS handles caching)
        # We construct random slices based on index, or sequential
        start = idx * self.block_size
        end = start + self.block_size + 1 # +1 because we need target
        
        # Grab chunk
        chunk = torch.from_numpy(self.data[start:end].astype(np.int64))
        
        # Inputs (x) and Targets (y)
        x = chunk[:-1]
        y = chunk[1:]
        
        # Create attention mask (all ones since we aren't padding)
        attention_mask = torch.ones_like(x)
        
        return x, attention_mask

# --- Usage in your Training Cell ---

# Point to the binary file you created in Step 1
train_ds = MemmapDataset("train_data/train.bin", block_size=128)
test_ds = MemmapDataset("test_data/test.bin", block_size=128)  

# Create DataLoader (standard)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
# Test it
for x, mask in train_loader:
    print("Input shape:", x.shape)
    print("Target shape:", x.shape) # Should match input
    break

In [ ]:
import pickle

# After successfully running:
# self.tokens = tokenizer.encode(raw_text)

# Save the tokens
with open('train_tokens.pkl', 'wb') as f:
    pickle.dump(train_ds.tokens, f)

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class GPT(nn.Module):
    def __init__(self, vocab_size, dim, n_layers, n_heads, block_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(block_size, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, n_heads) for _ in range(n_layers)
        ])
        self.ctrs = nn.ModuleList([
            CTR(num_slots=16, dim=dim, top_k=2) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(dim)
        self.fc_out = nn.Linear(dim, vocab_size)

    def forward(self, idx, attention_mask=None):
        # idx: (batch, seq_len), attention_mask: (batch, seq_len) optional
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        
        # Convert attention_mask to key_padding_mask for MultiheadAttention
        key_padding_mask = None
        if attention_mask is not None:
            # attention_mask has 1 for tokens, 0 for padding -> key_padding_mask expects True for pads
            key_padding_mask = attention_mask == 0
        
        for block in self.blocks:
            x = block(x, key_padding_mask=key_padding_mask)
        x = self.ln_f(x)
        logits = self.fc_out(x)
        return logits

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4*dim),
            nn.ReLU(),
            nn.Linear(4*dim, dim)
        )

    def forward(self, x, key_padding_mask=None):
        # x: (batch, seq_len, dim)
        # Build causal attn_mask (T, T) with -inf for disallowed positions
        attn_mask = self._causal_mask(x)
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x),
                                attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

    def _causal_mask(self, x):
        T = x.size(1)
        # attn_mask should be shape (T, T). Use large negative values for masked positions.
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn_mask = torch.zeros(T, T, device=x.device)
        attn_mask.masked_fill_(mask, float('-inf'))
        return attn_mask


class CTR(nn.Module):
    def __init__(self, num_slots: int, dim: int, top_k: int = 1):
        super().__init__()
        self.num_slots = num_slots
        self.top_k = top_k
        self.slot_proj = nn.Linear(dim, num_slots)  # token-to-slot scores
        self.ln = nn.LayerNorm(dim)
        self.slots = nn.Parameter(torch.randn(num_slots, dim))  # learnable slots

    def forward(self, x):
        # x: (batch, seq_len, dim)
        x_norm = self.ln(x)
        scores = self.slot_proj(x_norm)  # (batch, seq_len, num_slots)
        
        # Top-k routing
        topk_vals, topk_idx = torch.topk(scores, self.top_k, dim=-1)  # (batch, seq_len, top_k)
        weights = F.softmax(topk_vals, dim=-1)

        # Aggregate token information to slots
        batch, seq_len, dim = x.shape
        slot_updates = torch.zeros_like(self.slots)  # (num_slots, dim)
        for b in range(batch):
            for t in range(seq_len):
                for k in range(self.top_k):
                    idx = topk_idx[b, t, k]
                    slot_updates[idx] += weights[b, t, k] * x[b, t]
        
        # Update slots
        self.slots.data += slot_updates / batch  # simple moving average update

        return self.slots  # (num_slots, dim)

model = GPT(
    vocab_size=tokenizer.vocab_size,
    dim=512,
    n_layers=6,
    n_heads=8,
    block_size=128,
)


In [ ]:
#loss fn and optimizer
# Ensure pad_token_id is set
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, betas=(0.9, 0.95), weight_decay=0.1)


In [ ]:
def train():
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        batch_count = 0

        for batch in train_loader:
            input_ids, attention_mask = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            # Shift for causal LM: predict next token
            # inputs: (B, T-1), targets: (B, T-1)
            inputs = input_ids[:, :-1]
            targets = input_ids[:, 1:]
            attn_in = attention_mask[:, :-1]

            logits = model(inputs, attention_mask=attn_in)  # (B, T-1, vocab_size)

            # Reshape for CrossEntropyLoss
            logits = logits.reshape(-1, vocab_size)         # (B*(T-1), vocab_size)
            targets = targets.reshape(-1)                   # (B*(T-1),)

            loss = loss_fn(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            batch_count += 1

        avg_train_loss = total_loss / batch_count if batch_count > 0 else 0

        # ----- EVAL -----
        model.eval()
        eval_loss = 0
        eval_batch_count = 0
        with torch.inference_mode():
            for batch in test_loader:
                input_ids, attention_mask = batch
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                
                inputs = input_ids[:, :-1]
                targets = input_ids[:, 1:]
                attn_in = attention_mask[:, :-1]

                logits = model(inputs, attention_mask=attn_in)
                logits = logits.reshape(-1, vocab_size)
                targets = targets.reshape(-1)
                
                test_loss = loss_fn(logits, targets)
                eval_loss += test_loss.item()
                eval_batch_count += 1

        avg_eval_loss = eval_loss / eval_batch_count if eval_batch_count > 0 else 0

        if epoch % 1 == 0:
            print(f"Epoch {epoch} | Train loss: {avg_train_loss:.4f} | Test loss: {avg_eval_loss:.4f}")

epochs = 1
vocab_size = tokenizer.vocab_size

print("Starting training...")
model.to(device)
train()
print("Training complete!")


In [ ]:
# Save model checkpoint
import os

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "gpt_model.pt")

# Save model state dict and config
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "vocab_size": tokenizer.vocab_size,
    "dim": 512,
    "n_layers": 6,
    "n_heads": 8,
    "block_size": 128,
}

torch.save(checkpoint, checkpoint_path)
print(f"Model saved to {checkpoint_path}")


In [2]:
# Load model from checkpoint
def load_model(checkpoint_path, device):
    """Load model from saved checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Recreate model with same config
    model = GPT(
        vocab_size=checkpoint["vocab_size"],
        dim=checkpoint["dim"],
        n_layers=checkpoint["n_layers"],
        n_heads=checkpoint["n_heads"],
        block_size=checkpoint["block_size"]
    )
    
    # Load state dict
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    
    return model

# Load the saved model
loaded_model = load_model(checkpoint_path, device)
print(f"Model loaded from {checkpoint_path}")
print(f"Model parameters: {sum(p.numel() for p in loaded_model.parameters()):,}")


NameError: name 'checkpoint_path' is not defined

In [44]:
# Generate text using the trained model
def generate(model, prompt, tokenizer, device, max_tokens=50, temperature=1.0, top_k=50, use_greedy=True):
    """Generate text from a prompt using the trained model."""
    model.eval()
    
    try:
        # Encode the prompt
        input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
        print(f"Prompt encoded to {input_ids.shape[1]} tokens")
        
        with torch.inference_mode():
            for i in range(max_tokens):
                # Truncate to context window if needed
                if input_ids.shape[1] > 128:
                    input_ids = input_ids[:, -128:]
                
                # Get predictions for the last token
                logits = model(input_ids)  # (batch=1, seq_len, vocab_size)
                next_logits = logits[0, -1, :] / temperature  # (vocab_size,)
                
                if use_greedy:
                    # Greedy: take argmax
                    next_token_id = torch.argmax(next_logits, dim=-1)
                else:
                    # Top-k sampling
                    try:
                        top_k_val = min(top_k, next_logits.shape[0])
                        top_k_logits, top_k_idx = torch.topk(next_logits, top_k_val)
                        top_k_probs = torch.softmax(top_k_logits, dim=-1)
                        sampled_idx = torch.multinomial(top_k_probs, 1).squeeze()
                        next_token_id = top_k_idx[sampled_idx]
                    except Exception as e:
                        print(f"Sampling error: {e}. Falling back to greedy.")
                        next_token_id = torch.argmax(next_logits, dim=-1)
                
                # Append to sequence (reshape for concat)
                next_token_id = next_token_id.unsqueeze(0).unsqueeze(0)  # (1, 1)
                input_ids = torch.cat([input_ids, next_token_id], dim=1)
                
                # Stop if we generate end-of-sequence token
                if next_token_id.item() == tokenizer.eos_token_id:
                    break
        
        # Decode and return
        generated_text = tokenizer.decode(input_ids[0].tolist())
        return generated_text
    
    except Exception as e:
        print(f"Generation error: {e}")
        import traceback
        traceback.print_exc()
        return prompt

# Test generation with different prompts
print("=" * 60)
print("Testing text generation:")
print("=" * 60)

prompts = [

    "world"
]

# Use greedy by default (more stable)
for prompt in prompts:
    try:
        generated = generate(loaded_model, prompt, tokenizer, device, max_tokens=100, temperature=0.9, use_greedy=True)
        print(f"\nPrompt: '{prompt}'")
        print(f"Generated:\n{generated}\n")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 60)

# Optional: test sampling with better error handling
print("\n\nTesting with TOP-K SAMPLING (experimental):")
print("=" * 60)
for prompt in prompts[:1]:
    try:
        generated = generate(loaded_model, prompt, tokenizer, device, max_tokens=50, temperature=0.8, top_k=40, use_greedy=False)
        print(f"\nPrompt: '{prompt}'")
        print(f"Generated:\n{generated}\n")
    except Exception as e:
        print(f"Error: {e}")


Testing text generation:
Prompt encoded to 2 tokens

Prompt: 'world'
Generated:
<s> world around her.

One day, she saw a big, scary dog. She was scared and ran away. But then she saw a big dog. The dog was barking and wagging its tail. Lily was scared and ran away. She ran as fast as fast as she could, but the dog caught her and bit her.

Lily was scared and ran away. She ran away and ran, but the dog was too fast and caught her. She ran away and ran

------------------------------------------------------------


Testing with TOP-K SAMPLING (experimental):
Prompt encoded to 2 tokens

Prompt: 'world'
Generated:
<s> world around her. Everywhere she looked and saw birds, trees and birds. She loved to look at all of the trees and the trees and the birds.

One day, the sun went down and the girl went to the park with her friends


Prompt: 'world'
Generated:
<s> world around her.

One day, she saw a big, scary dog. She was scared and ran away. But then she saw a big dog. The dog was barking

In [1]:
# Load a checkpoint (.pt) from the `checkpoints/` folder and test it
import os
import glob
import random

checkpoint_dir = "checkpoints"
pattern = os.path.join(checkpoint_dir, "*.pt")
ckpts = sorted(glob.glob(pattern), key=os.path.getmtime)

if not ckpts:
    print(f"No checkpoint files found in {checkpoint_dir}. Place your .pt file(s) there.")
else:
    print(f"Found {len(ckpts)} checkpoint(s) in {checkpoint_dir}.")
    for i, p in enumerate(ckpts, 1):
        print(f"{i}: {p}")

    # Choose the checkpoint to load
    # Options: 'latest', 'random', or an index (1-based) from the printed list
    pick = 'latest'  # change to 'random' or an index like 1 if you want

    if isinstance(pick, int):
        chosen_path = ckpts[pick-1]
    elif pick == 'random':
        chosen_path = random.choice(ckpts)
    else:
        chosen_path = ckpts[-1]  # latest by modification time

    print(f"Loading checkpoint: {chosen_path}")

    try:
        loaded_model = load_model(chosen_path, device)
        print("Model loaded successfully.")
        print(f"Parameters: {sum(p.numel() for p in loaded_model.parameters()):,}")

        # Quick generation test
        test_prompt = "Once upon a time"
        print("\nGenerating sample text (greedy):")
        sample = generate(loaded_model, test_prompt, tokenizer, device, max_tokens=60, temperature=0.9, use_greedy=True)
        print(sample)

    except Exception as e:
        print(f"Failed to load or run the checkpoint: {e}")
        import traceback
        traceback.print_exc()


Found 2 checkpoint(s) in checkpoints.
1: checkpoints\gpt_model.pt
2: checkpoints\gpt_model (1).pt
Loading checkpoint: checkpoints\gpt_model (1).pt
Failed to load or run the checkpoint: name 'load_model' is not defined


Traceback (most recent call last):
  File "C:\Users\satyam prasad\AppData\Local\Temp\ipykernel_147080\2253620673.py", line 31, in <module>
    loaded_model = load_model(chosen_path, device)
                   ^^^^^^^^^^
NameError: name 'load_model' is not defined


In [ ]:


class CTR(nn.Module):
    def __init__(self, num_slots: int, dim: int, top_k: int = 1):
        super().__init__()
        self.num_slots = num_slots
        self.top_k = top_k
        self.slot_proj = nn.Linear(dim, num_slots)  # token-to-slot scores
        self.ln = nn.LayerNorm(dim)
        self.slots = nn.Parameter(torch.randn(num_slots, dim))  # learnable slots

    def forward(self, x):
        # x: (batch, seq_len, dim)
        x_norm = self.ln(x)
        scores = self.slot_proj(x_norm)  # (batch, seq_len, num_slots)
        
        # Top-k routing
        topk_vals, topk_idx = torch.topk(scores, self.top_k, dim=-1)  # (batch, seq_len, top_k)
        weights = F.softmax(topk_vals, dim=-1)

        # Aggregate token information to slots
        batch, seq_len, dim = x.shape
        slot_updates = torch.zeros_like(self.slots)  # (num_slots, dim)
        for b in range(batch):
            for t in range(seq_len):
                for k in range(self.top_k):
                    idx = topk_idx[b, t, k]
                    slot_updates[idx] += weights[b, t, k] * x[b, t]
        
        # Update slots
        self.slots.data += slot_updates / batch  # simple moving average update

        return self.slots  # (num_slots, dim)
